# Week 2: Text Cleaning: Before / After

Shows what `TextCleaner` (`scripts/text_cleaning.py`) changes, and proves it is safe.

**Privacy:** the examples in section 1 are hand-written strings and the project's own labeled queries. Real MLS remarks are used only for **aggregate counts**, never printed.

**Sections**
1. Before/after examples (invented remark-style strings and sample queries)
2. Noise removed from the real 1,000-remark sample
3. Effect on taxonomy matching (does cleaning help the Week 3 extractor?)
4. Safety checks (idempotence, protected acronyms, naive-regex comparison)


In [1]:
import json
import re
import sys
from pathlib import Path

import pandas as pd

ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "scripts"))

from text_cleaning import TextCleaner

DATA = ROOT / "data" / "processed"
cleaner = TextCleaner()

df = pd.read_csv(DATA / "listing_sample.csv")
df["remarks_clean"] = df["remarks"].map(cleaner.clean_text)
print(f"Loaded and cleaned {len(df)} remarks")

Loaded and cleaned 1000 remarks


## 1. Before / after examples

In [ ]:
pd.set_option("display.max_colwidth", 120)
pd.set_option("display.width", 200)

#invented strings that mimic the patterns profiled in docs/week2_profiling_report.md
examples = [
    ("Smart quotes & dashes",        "It\u2019s a \u201cmust-see\u201d\u2014call today!"),
    ("Invisible characters",         "Spacious\u200b\u200c home\u00a0with great light"),
    ("Mojibake",                     "Beautiful fa\u00c3\u00a7ade and pied-\u00c3\u00a0-terre"),
    ("Double spaces / newlines",     "Open  floor plan\nwith  a bonus room"),
    ("Markdown markers",             "***PRICE IMPROVEMENT***Welcome home"),
    ("Square footage",              "Approx. 2,450 sq  ft on a 7,200 SF lot"),
    ("Square footage + plus sign",   "Over 1,400+ sq/ft of living space"),
    ("Price shorthand",              "Offered at $1.15m, reduced from $1.2 million"),
    ("Bed/bath shorthand",           "Cozy 3BR/2BA with central A/C"),
    ("w/ and w/o",                   "Kitchen w/quartz island, sold w/o furniture"),
    ("Slash compounds",              "Indoor/outdoor living with pool/spa"),
    ("Abbreviations",                "LVP floors, new HVAC/AC, near hwy 101"),
    ("Protected acronyms",           "HOA $350/month, ADU, RV parking, EV charger, BBQ"),
]
table = pd.DataFrame(
    [(name, raw, cleaner.clean_text(raw)) for name, raw in examples],
    columns=["pattern", "before", "after"],
)
table

,pattern,before,after
0,Smart quotes & dashes,It’s a “must-see”—call today!,"It's a ""must-see"" - call today!"
1,Invisible characters,Spacious​‌ home with great light,Spacious home with great light
2,Mojibake,Beautiful faÃ§ade and pied-Ã -terre,Beautiful façade and pied-à-terre
3,Double spaces / newlines,Open floor plan\nwith a bonus room,Open floor plan with a bonus room
4,Markdown markers,***PRICE IMPROVEMENT***Welcome home,PRICE IMPROVEMENT Welcome home
5,Square footage,"Approx. 2,450 sq ft on a 7,200 SF lot",approximately 2450 square feet on a 7200 square feet lot
6,Square footage + plus sign,"Over 1,400+ sq/ft of living space",Over 1400+ square feet of living space
7,Price shorthand,"Offered at $1.15m, reduced from $1.2 million","Offered at $1150000, reduced from $1200000"
8,Bed/bath shorthand,Cozy 3BR/2BA with central A/C,Cozy 3 bedroom 2 bathroom with central air conditioning
9,w/ and w/o,"Kitchen w/quartz island, sold w/o furniture","Kitchen with quartz island, sold without furniture"


In [3]:
queries = pd.DataFrame(json.load(open(DATA / "sample_queries.json")))
queries["cleaned"] = queries["query"].map(cleaner.clean_text)
changed_queries = queries[queries["query"] != queries["cleaned"]][["query", "cleaned"]]
print(f"{len(changed_queries)} of {len(queries)} labeled queries change. Examples:")
changed_queries.head(10).reset_index(drop=True)

14 of 60 labeled queries change. Examples:


,query,cleaned
0,move-in ready 4 bed 3 bath in Temecula under 1.1m,move-in ready 4 bed 3 bath in Temecula under 1100000
1,3br 2ba w/ pool in Hemet under 550k ready now,3 bedroom 2 bathroom with pool in Hemet under 550000 ready now
2,single story homes in Corona with 3 car garage under $900k available now,single story homes in Corona with 3 car garage under $900000 available now
3,turnkey townhouse in Long Beach under 750k with attached garage,turnkey townhouse in Long Beach under 750000 with attached garage
4,new listings in San Diego with ADU and solar under 1.5m,new listings in San Diego with ADU and solar under 1500000
5,homes in Murrieta w/ no HOA and RV parking ready to tour,homes in Murrieta with no HOA and RV parking ready to tour
6,gated community homes in Palm Desert with pool under 800k open house sunday,gated community homes in Palm Desert with pool under 800000 open house sunday
7,"2,000 sqft or larger in Oceanside under $1.2M move in ready",2000 square feet or larger in Oceanside under $1200000 move in ready
8,just listed condos in Pasadena under 650k with parking,just listed condos in Pasadena under 650000 with parking
9,"investor special in Lancaster under 400k, cash offer ready","investor special in Lancaster under 400000, cash offer ready"


**What this shows:** shorthand that a search box or an agent produces (`3BR/2BA`, `450k`, `A/C`, `w/`) is mapped to one canonical form, protected acronyms are never touched, and risky patterns (an "SF" with no number, an "m" that means miles) are left alone.

## 2. Noise removed from the real sample


In [4]:
NOISE = {
    "curly quotes":              r"[\u2019\u2018\u201c\u201d]",
    "em/en dashes":              r"[\u2013\u2014]",
    "zero-width characters":     r"[\u200b\u200c\u200d]",
    "embedded newlines":         r"\n",
    "double spaces":             r"  ",
    "markdown ** markers":       r"\*\*",
    "bullets":                   r"\u2022",
    "'sq ft' style units":       r"(?i)\bsq\.?[ /]?ft\b|\bsqft\b",
    "'A/C' shorthand":           r"\bA/C\b",
    "'approx.' shorthand":       r"\bapprox\.",
    "'w/' shorthand":            r"\bw/",
    "$ + k/m price shorthand":   r"\$\d+(?:\.\d+)?\s?[kKmM]\b",
    "unrepaired mojibake":       r"[\u00c3\u00c2]",
}
rows = []
for name, pattern in NOISE.items():
    before = int(df["remarks"].str.contains(pattern, regex=True).sum())
    after = int(df["remarks_clean"].str.contains(pattern, regex=True).sum())
    rows.append((name, before, after))
noise = pd.DataFrame(rows, columns=["pattern", "remarks_before", "remarks_after"])
noise

,pattern,remarks_before,remarks_after
0,curly quotes,257,0
1,em/en dashes,204,0
2,zero-width characters,2,0
3,embedded newlines,232,0
4,double spaces,323,0
5,markdown ** markers,28,0
6,bullets,7,0
7,'sq ft' style units,165,0
8,'A/C' shorthand,35,0
9,'approx.' shorthand,9,0


In [5]:
changed = int((df["remarks"] != df["remarks_clean"]).sum())
print(f"Remarks changed: {changed} of {len(df)} ({changed / len(df):.1%})")
print(f"Average length: {df['remarks'].str.len().mean():.0f} -> {df['remarks_clean'].str.len().mean():.0f} characters")

Remarks changed: 832 of 1000 (83.2%)
Average length: 1321 -> 1319 characters


Every noise pattern in the table drops to zero. The average length barely moves, so the cleaner standardizes text rather than deleting content.

## 3. Does cleaning help taxonomy matching?


In [6]:
with open(DATA / "taxonomy.json") as f:
    tax = pd.DataFrame(json.load(f)["terms"])

def normalize(text):
    """Same helper as the Week 1 tests: lowercase, hyphens count as spaces."""
    return re.sub(r"\s+", " ", str(text).lower().replace("-", " ")).strip()

patterns = {term: re.compile(r"\b" + re.escape(normalize(term)) + r"\b") for term in tax["term"]}
raw_texts = df["remarks"].map(normalize).tolist()
clean_texts = df["remarks_clean"].map(normalize).tolist()

def term_counts(texts):
    return {term: sum(1 for t in texts if p.search(t)) for term, p in patterns.items()}

before_counts, after_counts = term_counts(raw_texts), term_counts(clean_texts)
total_before, total_after = sum(before_counts.values()), sum(after_counts.values())
print(f"Total term matches: {total_before} -> {total_after} ({(total_after - total_before) / total_before:+.1%})")

def coverage(texts):
    return sum(1 for t in texts if any(p.search(t) for p in patterns.values())) / len(texts)
print(f"Coverage (remarks with at least one term): {coverage(raw_texts):.1%} -> {coverage(clean_texts):.1%}")

gains = pd.DataFrame({"term": list(patterns), "before": list(before_counts.values()), "after": list(after_counts.values())})
gains["change"] = gains["after"] - gains["before"]
print("\nTerms with new matches:")
gains[gains["change"] != 0].sort_values("change", ascending=False).reset_index(drop=True)

Total term matches: 14612 -> 14760 (+1.0%)


Coverage (remarks with at least one term): 99.4% -> 99.4%

Terms with new matches:


,term,before,after,change
0,air conditioning,45,116,71
1,chef's kitchen,34,54,20
2,central air,30,50,20
3,luxury vinyl plank,26,39,13
4,vinyl plank flooring,30,39,9
5,indoor-outdoor living,49,56,7
6,butler's pantry,4,9,5
7,washer and dryer,63,64,1
8,living room,258,259,1
9,garage storage,2,3,1


Cleaning **adds** matches and removes none. Examples: `A/C` becomes "air conditioning", curly apostrophes no longer hide "chef's kitchen", and `LVP` becomes "luxury vinyl plank". Coverage was already near 99%, so the gain is in *depth* (more terms found per remark), which is what the Week 3 extractor needs.

## 4. Safety checks


In [ ]:
#idempotence
second_pass = df["remarks_clean"].map(cleaner.clean_text)
print("Remarks changed by a second pass:", int((second_pass != df["remarks_clean"]).sum()), "(must be 0)")

#protected acronyms must appear exactly as often after cleaning
protected = sorted(cleaner.protected)
def word_count(series, word):
    return int(series.str.count(r"(?i)\b" + word + r"\b").sum())
protected_check = pd.DataFrame(
    [(w, word_count(df["remarks"], w), word_count(df["remarks_clean"], w)) for w in protected],
    columns=["acronym", "before", "after"],
)
print("Protected acronyms whose count changed:", int((protected_check["before"] != protected_check["after"]).sum()), "(must be 0)")
protected_check.sort_values("before", ascending=False).head(8).reset_index(drop=True)

Remarks changed by a second pass: 0 (must be 0)


Protected acronyms whose count changed: 0 (must be 0)


,acronym,before,after
0,hoa,134,134
1,adu,124,124
2,rv,98,98
3,bbq,83,83
4,hvac,74,74
5,la,65,65
6,ev,47,47
7,led,31,31


In [ ]:
#the naive price regex vs the context-aware one, on invented trap strings
def naive_prices(text):
    text = re.sub(r"(\d+)k", lambda m: str(int(m.group(1)) * 1000), text, flags=re.I)
    return re.sub(r"(\d+\.?\d*)m", lambda m: str(int(float(m.group(1)) * 1_000_000)), text, flags=re.I)

traps = [
    "conventional or FHA 203K renovation loan",
    "beach is 0.2m away, about a 4min walk",
    "3 million visitors a year",
    "priced at 450k",
    "asking $1.001m",
]
pd.DataFrame(
    [(t, naive_prices(t), cleaner.normalize_prices(t)) for t in traps],
    columns=["input", "naive regex", "TextCleaner"],
)

,input,naive regex,TextCleaner
0,conventional or FHA 203K renovation loan,conventional or FHA 203000 renovation loan,conventional or FHA 203K renovation loan
1,"beach is 0.2m away, about a 4min walk","beach is 200000 away, about a 4000000in walk","beach is 0.2m away, about a 4min walk"
2,3 million visitors a year,3 million visitors a year,3 million visitors a year
3,priced at 450k,priced at 450000,priced at 450000
4,asking $1.001m,asking $1000999,asking $1001000


In [ ]:
#how many real remarks would the naive rule treat differently from the context-aware rule?
without_commas = df["remarks"].map(cleaner.normalize_numbers)
naive_changed = int((without_commas.map(naive_prices) != without_commas).sum())
differs = int((without_commas.map(naive_prices) != without_commas.map(cleaner.normalize_prices)).sum())
print(f"Naive rule alters {naive_changed} remarks; its output differs from TextCleaner's in {differs}.")

Naive rule alters 19 remarks; its output differs from TextCleaner's in 13.


## Summary
- **Coverage of noise:** every profiled noise pattern is eliminated (curly quotes, dashes, invisible characters, newlines, double spaces, markdown markers, sq ft variants, A/C, approx., w/, price shorthand).
- **Impact on taxonomy matching:** about 1% more matches and none lost; new hits come from `A/C`, curly apostrophes, `LVP` and slash compounds.
- **Safety:** cleaning is idempotent, protected acronyms are untouched, and context rules avoid the naive regex's false positives (loan names, distances, counts).
- **Tests:** `tests/test_week2.py` (159 cases) pins each rule, including every trap above.
- **Next (Week 3):** entity extraction can rely on canonical forms ("3 bedroom", "2000 square feet", "450000") instead of many spellings.
